# Application for ECATS - Optimisation

This document has been published for reproducing the application presented in a conference paper dedicated to AeroMAPS, focuding on the reproduction of ICAO LTAG scenarios. The different assumptions of this prospective scenario are given in the following. 

> **⚠ This notebook has been developed with AeroMAPS version v1.0.0 for obtaining the paper results. However, this notebook has been or could be modified in order to be executable with the latest versions of AeroMAPS, which sometimes leads to different results compared to the ones from the paper, due to some models' modifications. In order to retrieve the results of the paper, one can use the v1.0.0 version associated with the original notebook.**


This notebook extends the main examples_tsas_application by introducing a secario optimisation.

## Load

In [ ]:
%matplotlib widget
from aeromaps import create_process
from aeromaps.core.gemseo import CustomDataConverter
import gemseo as gm
from aeromaps.utils.functions import custom_logger_config

custom_logger_config(gm.configure_logger())

## Partitioning for international aviation

In [ ]:
# create_partitioning(
#     file="data/partitioning_data/aeroscope_international_data.csv", path="data/partitioning_data"
# )

## Process, data and compute

In [ ]:
process_is2medium = create_process(
    configuration_file="../data/data_optim/config_is2medium_no_degrowth.yaml",
    optimisation=True,
)

In [ ]:
# international_ask_share = 66.35 / 100


process_is2medium.parameters.generic_biomass_availability_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]


process_is2medium.parameters.generic_electricity_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]


process_is2medium.parameters.saf_ftg_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]
process_is2medium.parameters.saf_co2_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]
process_is2medium.parameters.lcaf_use_growth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]

process_is2medium.parameters.saf_ftg_no_degrowth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]
process_is2medium.parameters.saf_co2_no_degrowth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]
process_is2medium.parameters.lcaf_no_degrowth_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]

process_is2medium.parameters.blend_completeness_constraint_enforcement_years = [
    2035,
    2040,
    2045,
    2050,
    2055,
    2060,
    2065,
    2070,
]

process_is2medium.parameters.operations_contrails_final_gain = 60.0
process_is2medium.parameters.operations_contrails_start_year = 2030
process_is2medium.parameters.operations_contrails_duration = 10.0
process_is2medium.parameters.operations_contrails_final_overconsumption = 2.0

# Fixed values for the first mandate entries [2020, 2025, 2030]
# These are absolute shares, not fractions
process_is2medium.parameters.saf_ftg_mandate_share_values_fixed = [0, 0, 4.0]
process_is2medium.parameters.saf_co2_mandate_share_values_fixed = [0, 0, 4.0]
process_is2medium.parameters.lcaf_mandate_share_values_fixed = [0, 0, 4.0]

process_is2medium.parameters.rate_ramp_up_constraint_saf_ftg = 0.12
process_is2medium.parameters.rate_ramp_up_constraint_saf_co2 = 0.12
process_is2medium.parameters.rate_ramp_up_constraint_lcaf = 0.12

## Carbon budgets and Carbon Dioxide Removal [GtCO2]
process_is2medium.parameters.net_carbon_budget = 850.0
process_is2medium.parameters.carbon_dioxyde_removal_2100 = 285.0

### Optimisation problem setup with GEMSEO

In [ ]:
from gemseo.algos.design_space import DesignSpace


from gemseo.settings.opt import SLSQP_Settings


# Create a GEMSEO scenario


process_is2medium.gemseo_settings["scenario_type"] = "MDO"


# Define the design space: direct absolute shares (0-100) for SAF FTG, SAF CO2, and LCAF
# Targeted years for optimization variables: 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070
# The kerosene share is implicit: 100 - (FTG + CO2 + LCAF). Completeness is ensured via a constraint.


design_space = DesignSpace()

design_space.add_variable(
    "saf_co2_mandate_share_values_optim",
    size=8,
    lower_bound=[4.0] * 8,
    upper_bound=[100] * 8,
    value=[4.0, 4.1, 4.2, 4.3, 6.4, 10.5, 14.6, 20.0],
)


design_space.add_variable(
    "saf_ftg_mandate_share_values_optim",
    size=8,
    lower_bound=[4.0] * 8,
    upper_bound=[100] * 8,
    value=[4.0, 4.1, 4.2, 4.3, 6.4, 10.5, 14.6, 20.0],
)


design_space.add_variable(
    "lcaf_mandate_share_values_optim",
    size=8,
    lower_bound=[4.0] * 8,
    upper_bound=[100] * 8,
    value=[4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0],
)


process_is2medium.gemseo_settings["design_space"] = design_space


# Define the objective
objective_name = "temperature_increase_end_year"  # ,cumulative_co2_end_year, temperature_increase_end_year, mean_temperature_increase_from_aviation_2025_end
process_is2medium.gemseo_settings["objective_name"] = objective_name


process_is2medium.create_gemseo_scenario()
# Make optimisation objective values in the 1-10 interval
process_is2medium.scenario.formulation.optimization_problem.objective = (
    process_is2medium.scenario.formulation.optimization_problem.objective
)


# Add constraints (include blend_completeness_constraint with direct shares)
all_constraints = [
    # "biomass_trajectory_constraint",
    # "generic_electricity_trajectory_constraint",
    "saf_co2_use_growth_constraint",
    "saf_ftg_use_growth_constraint",
    "lcaf_use_growth_constraint",
    "saf_co2_no_degrowth_constraint",
    "saf_ftg_no_degrowth_constraint",
    "lcaf_no_degrowth_constraint",
    "blend_completeness_constraint",
]


for constraint in all_constraints:
    process_is2medium.scenario.add_constraint(constraint, constraint_type="ineq")


process_is2medium.scenario.set_differentiation_method("finite_differences")


slsqp_settings = SLSQP_Settings(
    max_iter=500,
    ftol_rel=0.0000001,
    ftol_abs=0.0000001,
    ineq_tolerance=0.05,
    normalize_design_space=True,
)


process_is2medium.gemseo_settings["algorithm"] = slsqp_settings  # cobyla_settings


# Adding design variables to the set of list types varaible (they are declared as ndarray but needed as lists within aeromaps functions)


CustomDataConverter._list_names.update(process_is2medium.scenario.get_optim_variable_names())

In [ ]:
import warnings

warnings.filterwarnings("ignore")
process_is2medium.compute()

In [ ]:
process_is2medium.scenario.post_process(
    post_name="OptHistoryView",
    save=False,
    show=True,
)

In [ ]:
from copy import deepcopy


# Extract optimization results
opt_result = process_is2medium.scenario.get_result().optimization_result.x_opt_as_dict

# Create a copy for validation
process_check = create_process(
    configuration_file="../data/data_optim/config_is2medium.yaml",
    optimisation=False,
)

process_check.parameters = deepcopy(process_is2medium.parameters)

# Apply optimized values to process_check parameters
process_check.parameters.saf_ftg_mandate_share_values_optim = opt_result[
    "saf_ftg_mandate_share_values_optim"
].tolist()
process_check.parameters.saf_co2_mandate_share_values_optim = opt_result[
    "saf_co2_mandate_share_values_optim"
].tolist()
process_check.parameters.lcaf_mandate_share_values_optim = opt_result[
    "lcaf_mandate_share_values_optim"
].tolist()

process_check.compute()

In [ ]:
process_check.write_json("results/slsqp_end_temp_no_degrowth_start2.json")

## Results

In [ ]:
process = process_check

In [ ]:
process.plot("fuel_shares")

In [ ]:
process.plot("air_transport_co2_emissions")

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals())